In [3]:
import time

import pandas as pd
import requests

# ----------------------
# 1. API 및 수집 설정
# ----------------------

url = "https://quotes.toscrape.com/api/quotes"

records = []
page = 1

# 예상치 못한 무한 반복 방지
max_pages = 20

# ----------------------
# 2. 여러 페이지 수집
# ----------------------

try:
    while True:
        if page > max_pages:
            raise RuntimeError("페이지 수 상한을 초과했습니다.")

        # 현재 페이지 요청
        response = requests.get(
            url,
            params={"page": page},
            timeout=10
        )

        response.raise_for_status()

        # JSON 응답 파싱
        data = response.json()
        quotes = data["quotes"]

        if not isinstance(quotes, list) or not quotes:
            raise ValueError("명언 목록이 없거나 형식이 잘못되었습니다.")

        # ----------------------
        # 3. 필요한 필드 추출
        # ----------------------

        for quote in quotes:
            records.append({
                "text": quote["text"],
                "author": quote["author"]["name"],
                "tags": ", ".join(quote["tags"]),
                "source_url": response.url
            })

        print(
            f"페이지 {page}: "
            f"{len(quotes)}개 수집 / "
            f"누적 {len(records)}개"
        )

        # ----------------------
        # 4. 다음 페이지 확인
        # ----------------------

        if not data["has_next"]:
            break

        page += 1
        time.sleep(0.5)

    # ----------------------
    # 5. DataFrame 생성
    # ----------------------

    columns = [
        "text",
        "author",
        "tags",
        "source_url"
    ]

    df = pd.DataFrame(
        records,
        columns=columns
    )

    # ----------------------
    # 6. CSV 저장
    # ----------------------

    file_path = "quotes_raw.csv"

    df.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig"
    )

    # ----------------------
    # 7. 저장 결과 확인
    # ----------------------

    loaded_df = pd.read_csv(
        file_path,
        encoding="utf-8-sig"
    )

    print("\n전체 수집 건수:", len(df))
    print("저장 후 행 수:", len(loaded_df))
    print("컬럼:", loaded_df.columns.tolist())
    print("저장 파일:", file_path)

    print("\n데이터 미리보기:")
    print(loaded_df.head())

except (requests.RequestException, ValueError, KeyError,
        TypeError, RuntimeError) as error:
    print("데이터 수집 중단:", error)
    print("이번 실행의 CSV 저장을 완료하지 못했습니다.")

페이지 1: 10개 수집 / 누적 10개
페이지 2: 10개 수집 / 누적 20개
페이지 3: 10개 수집 / 누적 30개
페이지 4: 10개 수집 / 누적 40개
페이지 5: 10개 수집 / 누적 50개
페이지 6: 10개 수집 / 누적 60개
페이지 7: 10개 수집 / 누적 70개
페이지 8: 10개 수집 / 누적 80개
페이지 9: 10개 수집 / 누적 90개
페이지 10: 10개 수집 / 누적 100개

전체 수집 건수: 100
저장 후 행 수: 100
컬럼: ['text', 'author', 'tags', 'source_url']
저장 파일: quotes_raw.csv

데이터 미리보기:
                                                text           author  \
0  “The world as we have created it is a process ...  Albert Einstein   
1  “It is our choices, Harry, that show what we t...     J.K. Rowling   
2  “There are only two ways to live your life. On...  Albert Einstein   
3  “The person, be it gentleman or lady, who has ...      Jane Austen   
4  “Imperfection is beauty, madness is genius and...   Marilyn Monroe   

                                           tags  \
0        change, deep-thoughts, thinking, world   
1                            abilities, choices   
2  inspirational, life, live, miracle, miracles   
3              ali

In [4]:
from pathlib import Path
import pandas as pd

# ----------------------
# 1. 수집 데이터 불러오기
# ----------------------

file_path = Path("quotes_raw.csv")

if not file_path.exists():
    raise FileNotFoundError(
        "quotes_raw.csv 파일이 없습니다. "
        "2장 API 수집 코드를 먼저 실행하세요."
    )

df = pd.read_csv(
    file_path,
    encoding="utf-8-sig"
)

# ----------------------
# 2. 기본 구조 확인
# ----------------------

print("데이터 Shape:", df.shape)
print("전체 행 수:", len(df))
print("컬럼 목록:", df.columns.tolist())

print("\n컬럼별 자료형:")
print(df.dtypes)

print("\nDataFrame 정보:")
df.info()

# ----------------------
# 3. 예상한 컬럼과 비교
# ----------------------

expected_columns = [
    "text",
    "author",
    "tags",
    "source_url"
]

missing_columns = [
    col for col in expected_columns
    if col not in df.columns
]

extra_columns = [
    col for col in df.columns
    if col not in expected_columns
]

print("\n누락된 컬럼:", missing_columns)
print("추가된 컬럼:", extra_columns)

# ----------------------
# 4. 실제 데이터 일부 확인
# ----------------------

print("\n데이터 미리보기:")
print(df.head(3))

# ----------------------
# 5. 수집 출처 확인
# ----------------------

if "source_url" in df.columns:
    print("\n출처 URL별 데이터 건수:")
    print(df["source_url"].value_counts(dropna=False))

# ----------------------
# 6. 구조 확인 결과
# ----------------------

if df.empty:
    print("\n확인 필요: 데이터가 비어 있습니다.")

elif missing_columns or extra_columns:
    print("\n확인 필요: 예상한 컬럼 구조와 다릅니다.")

else:
    print("\n기본 행·컬럼 구조 확인 완료")

데이터 Shape: (100, 4)
전체 행 수: 100
컬럼 목록: ['text', 'author', 'tags', 'source_url']

컬럼별 자료형:
text          str
author        str
tags          str
source_url    str
dtype: object

DataFrame 정보:
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   text        100 non-null    str  
 1   author      100 non-null    str  
 2   tags        97 non-null     str  
 3   source_url  100 non-null    str  
dtypes: str(4)
memory usage: 23.4 KB

누락된 컬럼: []
추가된 컬럼: []

데이터 미리보기:
                                                text           author  \
0  “The world as we have created it is a process ...  Albert Einstein   
1  “It is our choices, Harry, that show what we t...     J.K. Rowling   
2  “There are only two ways to live your life. On...  Albert Einstein   

                                           tags  \
0        change, deep-thoughts, thinking, world   
1                

In [ ]:
df.shape
df.tail(3)


,text,author,tags,source_url
97,“Never tell the truth to people who are not wo...,Mark Twain,truth,https://quotes.toscrape.com/api/quotes?page=10
98,"“A person's a person, no matter how small.”",Dr. Seuss,inspirational,https://quotes.toscrape.com/api/quotes?page=10
99,“... a mind needs books as a sword needs a whe...,George R.R. Martin,"books, mind",https://quotes.toscrape.com/api/quotes?page=10


In [7]:
df.dtypes

text          str
author        str
tags          str
source_url    str
dtype: object

In [8]:
df.info

<bound method DataFrame.info of                                                  text              author  \
0   “The world as we have created it is a process ...     Albert Einstein   
1   “It is our choices, Harry, that show what we t...        J.K. Rowling   
2   “There are only two ways to live your life. On...     Albert Einstein   
3   “The person, be it gentleman or lady, who has ...         Jane Austen   
4   “Imperfection is beauty, madness is genius and...      Marilyn Monroe   
..                                                ...                 ...   
95  “You never really understand a person until yo...          Harper Lee   
96  “You have to write the book that wants to be w...   Madeleine L'Engle   
97  “Never tell the truth to people who are not wo...          Mark Twain   
98        “A person's a person, no matter how small.”           Dr. Seuss   
99  “... a mind needs books as a sword needs a whe...  George R.R. Martin   

                                           

In [10]:
df.isna().sum()

text          0
author        0
tags          3
source_url    0
dtype: int64

In [11]:
df.columns

Index(['text', 'author', 'tags', 'source_url'], dtype='str')

In [12]:
from pathlib import Path
import pandas as pd

# ----------------------
# 1. CSV 파일 불러오기
# ----------------------

file_path = Path("quotes_raw.csv")

if not file_path.exists():
    raise FileNotFoundError(
        "quotes_raw.csv 파일이 없습니다."
    )

df = pd.read_csv(
    file_path,
    encoding="utf-8-sig",
    keep_default_na=False
)

# ----------------------
# 2. 필수 컬럼 확인
# ----------------------

expected_columns = [
    "text",
    "author",
    "tags",
    "source_url"
]

missing_columns = [
    col for col in expected_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"누락된 컬럼: {missing_columns}"
    )

print("전체 데이터 건수:", len(df))
print("컬럼:", df.columns.tolist())

# ----------------------
# 3. 결측값과 빈 문자열 확인
# ----------------------

# 원본을 유지하고 확인용 복사본 생성
check_df = df.copy()

for col in expected_columns:
    # 문자열 양쪽 공백 제거
    check_df[col] = (
        check_df[col]
        .astype("string")
        .str.strip()
    )

    # 빈 문자열을 결측값으로 표시
    check_df.loc[
        check_df[col] == "", col
    ] = pd.NA

print("\n컬럼별 결측·빈 문자열 건수:")
print(check_df[expected_columns].isna().sum())

# ----------------------
# 4. 중복 데이터 확인
# ----------------------

# 본문과 작성자가 모두 있는 행만 비교
valid_mask = (
    check_df["text"].notna()
    & check_df["author"].notna()
)

duplicate_mask = (
    valid_mask
    & check_df.duplicated(
        subset=["text", "author"],
        keep=False
    )
)

print("\n중복 후보 행 수:", duplicate_mask.sum())

print("\n중복 후보 데이터:")
print(
    check_df.loc[
        duplicate_mask,
        ["text", "author"]
    ].head(10)
)

# ----------------------
# 5. 수집 출처별 건수 확인
# ----------------------

print("\n수집 출처별 건수:")
print(
    check_df["source_url"].value_counts(
        dropna=False
    ).sort_index()
)

# ----------------------
# 6. 전체 확인 결과
# ----------------------

print("\n품질 탐색 결과")

print("전체 행 수:", len(check_df))

print(
    "본문 누락:",
    check_df["text"].isna().sum()
)

print(
    "작성자 누락:",
    check_df["author"].isna().sum()
)

print(
    "중복 후보:",
    duplicate_mask.sum()
)

print("\n원본 데이터는 변경하지 않았습니다.")

전체 데이터 건수: 100
컬럼: ['text', 'author', 'tags', 'source_url']

컬럼별 결측·빈 문자열 건수:
text          0
author        0
tags          3
source_url    0
dtype: int64

중복 후보 행 수: 0

중복 후보 데이터:
Empty DataFrame
Columns: [text, author]
Index: []

수집 출처별 건수:
source_url
https://quotes.toscrape.com/api/quotes?page=1     10
https://quotes.toscrape.com/api/quotes?page=10    10
https://quotes.toscrape.com/api/quotes?page=2     10
https://quotes.toscrape.com/api/quotes?page=3     10
https://quotes.toscrape.com/api/quotes?page=4     10
https://quotes.toscrape.com/api/quotes?page=5     10
https://quotes.toscrape.com/api/quotes?page=6     10
https://quotes.toscrape.com/api/quotes?page=7     10
https://quotes.toscrape.com/api/quotes?page=8     10
https://quotes.toscrape.com/api/quotes?page=9     10
Name: count, dtype: int64[pyarrow]

품질 탐색 결과
전체 행 수: 100
본문 누락: 0
작성자 누락: 0
중복 후보: 0

원본 데이터는 변경하지 않았습니다.
